# 07_01 — Preparação da ABT Early-Stage

## Objetivo

Preparar a primeira base **early-stage** para o modelo de classificação `target_banco_ganhou`.

Este notebook executa somente a **Etapa 1** do pipeline:

1. Carregar a ABT com target.
2. Validar `target_banco_ganhou`.
3. Remover colunas com leakage.
4. Remover variáveis bloqueadas para o modelo early-stage.
5. Criar apenas features linha a linha, sem agregações históricas.
6. Fazer split temporal 80/20 usando `Datainicial`.
7. Salvar `df_dev`, `df_holdout`, lista de features e relatórios de auditoria.

## Importante

Nesta etapa ainda **não** criamos:

- taxa histórica de perda;
- target encoding;
- WoE;
- OptBinning supervisionado;
- frequência calculada no dataset inteiro;
- agregações por grupo;
- calibração;
- conformal prediction.

Essas etapas virão depois para evitar leakage temporal.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 240)

BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
REPORTS_DIR = BASE_DIR / "outputs" / "reports" / "modelagem_early_stage"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = PROCESSED_DIR / "abt_eda_valorajuizado_target_banco_ganhou.parquet"

TARGET_COL = "target_banco_ganhou"
DATE_COL = "Datainicial"
VALUE_COL_ORIGINAL = "Valorajuizado"

RANDOM_STATE = 42

print("Setup concluído.")

## 2. Carregar base

In [ ]:
if INPUT_FILE.exists():
    df = pd.read_parquet(INPUT_FILE)
    print(f"Arquivo carregado: {INPUT_FILE}")
else:
    raise FileNotFoundError(
        f"Arquivo não encontrado: {INPUT_FILE}. "
        "Confirme se o notebook anterior salvou a ABT nesse caminho."
    )

print("Shape inicial:", df.shape)
df.head()

## 3. Funções auxiliares

In [ ]:
NULL_STRINGS = {
    "", "null", "nan", "none", "na", "n/a", "-",
    "não informado", "nao informado", "sem informação", "sem informacao",
    "não se aplica", "nao se aplica"
}


def normalize_text(x):
    if pd.isna(x):
        return None

    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKD", x)
    x = "".join(c for c in x if not unicodedata.combining(c))
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()

    if x in NULL_STRINGS:
        return None

    return x


def clean_colname(name):
    name = str(name)
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r"[^a-zA-Z0-9_]", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_").lower()


def existing_cols(df, cols):
    return [c for c in cols if c in df.columns]


def save_report(df_report, filename):
    path = REPORTS_DIR / filename
    df_report.to_csv(path, index=False)
    print(f"Relatório salvo em: {path}")


def parse_sim_nao(x):
    x = normalize_text(x)

    if x in ["sim", "s", "yes", "true", "1"]:
        return 1

    if x in ["nao", "não", "n", "no", "false", "0"]:
        return 0

    return np.nan


def safe_log1p(s):
    s = pd.to_numeric(s, errors="coerce")
    return np.log1p(s.clip(lower=0))


def winsorize_by_train_quantiles(train_s, apply_s, lower_q=0.01, upper_q=0.99):
    """
    Calcula limites no treino/dev e aplica em outra série.
    Nesta etapa, usamos essa função depois do split.
    """
    train_s = pd.to_numeric(train_s, errors="coerce")
    apply_s = pd.to_numeric(apply_s, errors="coerce")

    lower = train_s.quantile(lower_q)
    upper = train_s.quantile(upper_q)

    return apply_s.clip(lower, upper), lower, upper

## 4. Validação mínima da base

In [ ]:
required_cols = [
    TARGET_COL,
    DATE_COL,
    VALUE_COL_ORIGINAL,
]

missing_required = [c for c in required_cols if c not in df.columns]

if missing_required:
    raise KeyError(f"Colunas obrigatórias ausentes: {missing_required}")

df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df[VALUE_COL_ORIGINAL] = pd.to_numeric(df[VALUE_COL_ORIGINAL], errors="coerce")

df = df[df[TARGET_COL].notna()].copy()
df[TARGET_COL] = df[TARGET_COL].astype(int)

df = df[df[TARGET_COL].isin([0, 1])].copy()
df = df[df[DATE_COL].notna()].copy()
df = df[df[VALUE_COL_ORIGINAL].notna()].copy()
df = df[df[VALUE_COL_ORIGINAL] > 0].copy()

print("Shape após validações:", df.shape)

target_dist = (
    df.groupby(TARGET_COL)
    .size()
    .reset_index(name="qtd")
)

target_dist["classe"] = target_dist[TARGET_COL].map({
    0: "banco_perdeu",
    1: "banco_ganhou"
})

target_dist["perc"] = target_dist["qtd"] / target_dist["qtd"].sum()

save_report(target_dist, "01_target_distribution_early_stage.csv")
target_dist

## 5. Definir colunas de leakage

Essas colunas não podem entrar no modelo early-stage, pois carregam informação do desfecho, da sentença, da condenação, de acordos ou de predições externas pós-processuais.

In [ ]:
leakage_exact_cols = [
    # Target e derivados
    "target_banco_ganhou",
    "target_banco_perdeu",
    "desfecho_categoria",
    "valor_perdido",
    "valor_ganho",

    # Usadas para construir o target
    "Situacao",
    "Motivo_Encerramento",
    "Dataencerramento",

    # Resultados, sentença, condenação, acordo
    "resultado_final_processo_text",
    "resultadojulgamento_tipo",
    "resultado_do_recurso_texto",
    "resultado_exec_texto",
    "resultado_recurso_original_texto",
    "resultado_original_text",
    "resultado_acordao",
    "sentenca_tipo",
    "sentenca_data",
    "sentenca_concluso_data",
    "sentenca_duracao",
    "texto_sentenca_texto",
    "texto_tutela_texto",
    "tipo_de_decisao_texto",
    "tipo_de_extincao_texto",

    # Valores pós-desfecho
    "condenacao_valor",
    "condenacao_dano_material_valor",
    "condenacao_pensao_valor",
    "valor_do_acordo_valor",
    "valor_arbitrado_valor",
    "honorario_valor",

    # Datas claramente pós-evento decisivo
    "data_condenacao_data",
    "data_sentenca_execucao_data",
    "data_execucao_sentenca_data",
    "data_fim_da_execucao_data",
    "data_do_acordao_data",
    "data_do_acordo_qqerfase_data",
    "data_acordo_homologado_antes_sentenca_data",
    "data_transito_julgado_data",
    "data_transito_julgado_extinto_ou_improcedente_data",
    "data_transito_julgado_procedente_ou_parc_procedente_data",
    "transitojulgado_data",
    "transitojulg_imp_ext_data",
    "transitojulgparcproc_data",

    # Predições externas DeepLegal
    "procedente_prob",
    "procedente_parc_procedente_prob",
    "condenacao_valor_prob",
    "dano_moral_prob",
    "acordo_prob",
    "provavel_dias_faltantes_sentenca_string",
    "provavel_dias_ate_sentenca_string",
    "provavel_valor_condenacao_string",
    "provavel_valor_dano_moral_string",
    "provavel_dias_ate_arquivamento_string",
    "provavel_dias_ate_alvara_string",
    "provavel_dias_ate_execucao_string",
]

leakage_patterns = [
    r"^target_",
    r"^desfecho",
    r"^valor_perdido",
    r"^valor_ganho",
    r"resultado",
    r"sentenca",
    r"condenacao",
    r"acordo_prob",
    r"procedente_prob",
    r"procedente_parc",
    r"dano_moral_prob",
    r"provavel_",
    r"texto_sentenca",
    r"texto_tutela",
    r"transito",
    r"data_condenacao",
    r"data_sentenca",
    r"data_execucao",
    r"valor_do_acordo",
    r"valor_arbitrado",
]


def is_leakage_col(col):
    if col in leakage_exact_cols:
        return True

    c = str(col).lower()

    for pattern in leakage_patterns:
        if re.search(pattern, c):
            return True

    return False


leakage_cols_found = [c for c in df.columns if is_leakage_col(c)]

leakage_report = pd.DataFrame({
    "coluna": leakage_cols_found,
    "tipo_remocao": "leakage_ou_pos_desfecho"
})

save_report(leakage_report, "02_leakage_cols_removed_early_stage.csv")

print("Qtd colunas removidas por leakage:", len(leakage_cols_found))
leakage_report.head(100)

## 6. Definir variáveis bloqueadas para early-stage

Essas colunas podem ser úteis em um score atual da carteira, mas são bloqueadas para um modelo que simula previsão no início do processo, salvo validação futura de que existiam exatamente na `Datainicial`.

In [ ]:
blocked_early_stage_cols = [
    # Gestão/acompanhamento do processo
    "Fasedoprocesso",
    "fase_processual_texto",
    "status_texto",
    "status_tipo",
    "Qtddias_Ultimoandamento",

    # Decisões internas ao longo do ciclo
    "Passiveldeacordo",
    "Processorelevante",
    "Estimativa_De_Perda",

    # Responsáveis podem ser posteriores ao cadastro inicial
    "Credenciado",
    "Adv_Interno",
    "Advogadoadverso",

    # Campos operacionais potencialmente posteriores
    "audiencia_data",
    "audiencia_designada_data",
    "citacao_data",
    "citacao_conf_data",
    "data_da_contestacao_data",
    "data_do_recurso_data",
    "data_envio_do_recurso_data",
    "data_retorno_do_recurso_data",
    "retorno_laudo_pericial_data",
    "penhorarequerida_data",
    "penhoralevantada_data",
    "penhora_conf_data",
]

blocked_early_stage_cols_found = [
    c for c in blocked_early_stage_cols
    if c in df.columns
]

blocked_report = pd.DataFrame({
    "coluna": blocked_early_stage_cols_found,
    "tipo_remocao": "bloqueada_para_modelo_early_stage"
})

save_report(blocked_report, "03_blocked_early_stage_cols.csv")

print("Qtd colunas bloqueadas para early-stage:", len(blocked_early_stage_cols_found))
blocked_report

## 7. Colunas permitidas no early-stage

Aqui definimos o conjunto bruto de colunas candidatas.

Observação: campos como `valor_valor` da DeepLegal devem ser usados apenas se forem valor inicial do processo. Se esse valor for enriquecido posteriormente, remova-o.

In [ ]:
allowed_raw_cols = [
    # Identificadores úteis para rastreio, mas não entram como feature
    "Numerodistribuicao",
    "Identificador",

    # Target e data
    TARGET_COL,
    DATE_COL,

    # Financeiro inicial
    "Valorajuizado",
    "valor_ajuizado",

    # Cadastro Benner
    "Carteira",
    "Nm_Produto",
    "Nm_Acao",
    "Nm_Assunto",
    "Tipoprocesso",
    "Uf",
    "Comarca",
    "Nm_Empresa",
    "Denominacao",

    # DeepLegal estrutural, assumindo que está disponível no início
    "classe_texto",
    "area_texto",
    "assunto_normalizado_texto",
    "assunto_texto",
    "cidade",
    "uf",
    "vara_texto",
    "orgao_julgador_texto",
    "tipo_de_processo_texto",
    "tipo_de_justica_texto",
    "tipo_de_requerente_texto",
    "tipo_de_requerido_texto",

    # Valor externo, usar com cautela se for valor inicial do processo
    "valor_valor",
]

allowed_raw_cols = existing_cols(df, allowed_raw_cols)

cols_to_keep = []

for col in allowed_raw_cols:
    if col in [TARGET_COL, DATE_COL, "Numerodistribuicao", "Identificador"]:
        cols_to_keep.append(col)
        continue

    if col in leakage_cols_found:
        continue

    if col in blocked_early_stage_cols_found:
        continue

    cols_to_keep.append(col)

df_base = df[cols_to_keep].copy()

print("Shape df_base:", df_base.shape)
print("Colunas mantidas:", len(df_base.columns))
df_base.head()

## 8. Relatório de colunas mantidas/removidas

In [ ]:
all_cols = set(df.columns)
kept_cols = set(df_base.columns)
removed_cols = all_cols - kept_cols

removal_reason = []

for col in sorted(all_cols):
    if col in kept_cols:
        reason = "mantida"
    elif col in leakage_cols_found:
        reason = "removida_leakage"
    elif col in blocked_early_stage_cols_found:
        reason = "removida_bloqueada_early_stage"
    else:
        reason = "fora_lista_permitida"

    removal_reason.append({
        "coluna": col,
        "acao": reason
    })

columns_audit = pd.DataFrame(removal_reason)

save_report(columns_audit, "04_columns_audit_early_stage.csv")
columns_audit["acao"].value_counts()

## 9. Criar features financeiras row-wise

In [ ]:
df_fe = df_base.copy()

if "valor_ajuizado" in df_fe.columns:
    df_fe["fe_valor_ajuizado"] = pd.to_numeric(df_fe["valor_ajuizado"], errors="coerce")
elif "Valorajuizado" in df_fe.columns:
    df_fe["fe_valor_ajuizado"] = pd.to_numeric(df_fe["Valorajuizado"], errors="coerce")
else:
    raise KeyError("Nenhuma coluna de valor encontrada: valor_ajuizado ou Valorajuizado")

df_fe["fe_valor_ajuizado_log1p"] = safe_log1p(df_fe["fe_valor_ajuizado"])

bins_valor = [
    -np.inf,
    1_000,
    5_000,
    10_000,
    25_000,
    50_000,
    100_000,
    250_000,
    500_000,
    1_000_000,
    np.inf
]

labels_valor = [
    "ate_1k",
    "1k_5k",
    "5k_10k",
    "10k_25k",
    "25k_50k",
    "50k_100k",
    "100k_250k",
    "250k_500k",
    "500k_1mm",
    "acima_1mm"
]

df_fe["fe_faixa_valor_ajuizado"] = pd.cut(
    df_fe["fe_valor_ajuizado"],
    bins=bins_valor,
    labels=labels_valor,
    include_lowest=True
).astype(str)

df_fe["fe_flag_valor_ate_10k"] = (df_fe["fe_valor_ajuizado"] <= 10_000).astype(int)
df_fe["fe_flag_valor_10k_50k"] = (
    (df_fe["fe_valor_ajuizado"] > 10_000) &
    (df_fe["fe_valor_ajuizado"] <= 50_000)
).astype(int)
df_fe["fe_flag_valor_acima_50k"] = (df_fe["fe_valor_ajuizado"] > 50_000).astype(int)
df_fe["fe_flag_valor_acima_100k"] = (df_fe["fe_valor_ajuizado"] > 100_000).astype(int)

financial_features = [
    "fe_valor_ajuizado",
    "fe_valor_ajuizado_log1p",
    "fe_faixa_valor_ajuizado",
    "fe_flag_valor_ate_10k",
    "fe_flag_valor_10k_50k",
    "fe_flag_valor_acima_50k",
    "fe_flag_valor_acima_100k",
]

df_fe[financial_features].head()

## 10. Criar features temporais row-wise

In [ ]:
df_fe[DATE_COL] = pd.to_datetime(df_fe[DATE_COL], errors="coerce")

df_fe["fe_ano_inicial"] = df_fe[DATE_COL].dt.year
df_fe["fe_mes_inicial"] = df_fe[DATE_COL].dt.month
df_fe["fe_trimestre_inicial"] = df_fe[DATE_COL].dt.quarter
df_fe["fe_semestre_inicial"] = np.where(df_fe["fe_mes_inicial"] <= 6, 1, 2)
df_fe["fe_dia_semana_inicial"] = df_fe[DATE_COL].dt.dayofweek

df_fe["fe_cohort_pre_2020"] = (df_fe["fe_ano_inicial"] < 2020).astype(int)
df_fe["fe_cohort_2020_2021"] = df_fe["fe_ano_inicial"].between(2020, 2021).astype(int)
df_fe["fe_cohort_2022_2023"] = df_fe["fe_ano_inicial"].between(2022, 2023).astype(int)
df_fe["fe_cohort_pos_2024"] = (df_fe["fe_ano_inicial"] >= 2024).astype(int)

temporal_features = [
    "fe_ano_inicial",
    "fe_mes_inicial",
    "fe_trimestre_inicial",
    "fe_semestre_inicial",
    "fe_dia_semana_inicial",
    "fe_cohort_pre_2020",
    "fe_cohort_2020_2021",
    "fe_cohort_2022_2023",
    "fe_cohort_pos_2024",
]

df_fe[temporal_features].head()

## 11. Normalizar categóricas permitidas

In [ ]:
categorical_raw_cols = existing_cols(df_fe, [
    "Carteira",
    "Nm_Produto",
    "Nm_Acao",
    "Nm_Assunto",
    "Tipoprocesso",
    "Uf",
    "Comarca",
    "Nm_Empresa",
    "Denominacao",
    "classe_texto",
    "area_texto",
    "assunto_normalizado_texto",
    "assunto_texto",
    "cidade",
    "uf",
    "vara_texto",
    "orgao_julgador_texto",
    "tipo_de_processo_texto",
    "tipo_de_justica_texto",
    "tipo_de_requerente_texto",
    "tipo_de_requerido_texto",
])

categorical_norm_features = []

for col in categorical_raw_cols:
    new_col = f"fe_{clean_colname(col)}_norm"
    df_fe[new_col] = df_fe[col].apply(normalize_text)
    categorical_norm_features.append(new_col)

print("Qtd categóricas normalizadas:", len(categorical_norm_features))
categorical_norm_features

## 12. Criar interações categóricas simples

In [ ]:
interaction_specs = [
    ("Nm_Produto", "Nm_Acao"),
    ("Nm_Produto", "Nm_Assunto"),
    ("Nm_Acao", "Nm_Assunto"),
    ("Nm_Produto", "Nm_Acao", "Nm_Assunto"),
    ("Uf", "Comarca"),
    ("Uf", "Nm_Assunto"),
    ("Comarca", "Nm_Assunto"),
    ("Carteira", "Nm_Assunto"),
    ("classe_texto", "assunto_normalizado_texto"),
    ("tipo_de_justica_texto", "assunto_normalizado_texto"),
]

interaction_features = []

for spec in interaction_specs:
    if all(c in df_fe.columns for c in spec):
        new_col = "fe_inter_" + "__".join(clean_colname(c) for c in spec)

        df_fe[new_col] = (
            df_fe[list(spec)]
            .astype(str)
            .fillna("nao_informado")
            .agg(" | ".join, axis=1)
            .apply(normalize_text)
        )

        interaction_features.append(new_col)

print("Qtd interações criadas:", len(interaction_features))
interaction_features

## 13. Features de qualidade cadastral

In [ ]:
cadastro_cols_for_missing = existing_cols(df_fe, [
    "Carteira",
    "Nm_Produto",
    "Nm_Acao",
    "Nm_Assunto",
    "Tipoprocesso",
    "Uf",
    "Comarca",
    "Nm_Empresa",
    "classe_texto",
    "assunto_normalizado_texto",
    "tipo_de_justica_texto",
])

for col in cadastro_cols_for_missing:
    norm_col = f"fe_{clean_colname(col)}_norm"

    if norm_col in df_fe.columns:
        df_fe[f"fe_flag_missing_{clean_colname(col)}"] = df_fe[norm_col].isna().astype(int)

missing_flag_features = [
    c for c in df_fe.columns
    if c.startswith("fe_flag_missing_")
]

df_fe["fe_qtd_campos_cadastro_missing"] = df_fe[missing_flag_features].sum(axis=1)
df_fe["fe_pct_campos_cadastro_missing"] = (
    df_fe["fe_qtd_campos_cadastro_missing"] / max(len(missing_flag_features), 1)
)

quality_features = missing_flag_features + [
    "fe_qtd_campos_cadastro_missing",
    "fe_pct_campos_cadastro_missing",
]

df_fe[quality_features].head()

## 14. Features de keywords jurídicas simples

In [ ]:
text_cols_for_keywords = existing_cols(df_fe, [
    "Nm_Produto",
    "Nm_Acao",
    "Nm_Assunto",
    "classe_texto",
    "area_texto",
    "assunto_normalizado_texto",
    "assunto_texto",
    "tipo_de_processo_texto",
    "tipo_de_justica_texto",
])

df_fe["fe_texto_inicial_curto"] = (
    df_fe[text_cols_for_keywords]
    .astype(str)
    .fillna("")
    .agg(" ".join, axis=1)
    .apply(lambda x: normalize_text(x) or "")
)

keyword_groups = {
    "fraude": ["fraude", "suspeita", "transacao suspeita", "golpe"],
    "protesto": ["protesto", "protesto indevido"],
    "negativacao": ["negativacao", "restricao", "serasa", "spc"],
    "dano_moral": ["dano moral", "moral"],
    "revisional": ["revisional", "juros", "tarifa", "contrato"],
    "cobranca": ["cobranca", "cobrança", "divida", "debito"],
    "cartao": ["cartao", "cartão"],
    "conta": ["conta", "corrente", "poupanca", "poupança"],
    "consignado": ["consignado"],
    "imobiliario": ["imobiliario", "imobiliário", "imovel", "imóvel"],
    "pix": ["pix"],
    "acesso": ["acesso", "senha", "bloqueio"],
}

keyword_features = []

for key, terms in keyword_groups.items():
    col = f"fe_kw_{key}"

    pattern = "|".join([re.escape(normalize_text(t) or t) for t in terms])

    df_fe[col] = (
        df_fe["fe_texto_inicial_curto"]
        .str.contains(pattern, na=False, regex=True)
        .astype(int)
    )

    keyword_features.append(col)

text_features = ["fe_texto_inicial_curto"] + keyword_features

df_fe[text_features].head()

## 15. Split temporal 80/20

Este é o holdout final. Depois dele, o holdout não deve ser usado para decisões de feature engineering, tuning ou escolha de modelo.

In [ ]:
df_fe = df_fe.sort_values(DATE_COL).copy()

split_date = df_fe[DATE_COL].quantile(0.80)

df_dev = df_fe[df_fe[DATE_COL] <= split_date].copy()
df_holdout = df_fe[df_fe[DATE_COL] > split_date].copy()

split_report = pd.DataFrame([{
    "split_date": split_date,
    "qtd_total": len(df_fe),
    "qtd_dev": len(df_dev),
    "qtd_holdout": len(df_holdout),
    "perc_dev": len(df_dev) / len(df_fe),
    "perc_holdout": len(df_holdout) / len(df_fe),
    "taxa_target_dev": df_dev[TARGET_COL].mean(),
    "taxa_target_holdout": df_holdout[TARGET_COL].mean(),
    "valor_medio_dev": df_dev["fe_valor_ajuizado"].mean(),
    "valor_medio_holdout": df_holdout["fe_valor_ajuizado"].mean(),
    "data_min_dev": df_dev[DATE_COL].min(),
    "data_max_dev": df_dev[DATE_COL].max(),
    "data_min_holdout": df_holdout[DATE_COL].min(),
    "data_max_holdout": df_holdout[DATE_COL].max(),
}])

save_report(split_report, "05_temporal_holdout_split_report.csv")
split_report

## 16. Winsorização usando apenas o dev/train

Os limites de winsorização são aprendidos no `df_dev` e aplicados no `df_holdout`.

In [ ]:
df_dev["fe_valor_ajuizado_winsor"], lower_winsor, upper_winsor = winsorize_by_train_quantiles(
    train_s=df_dev["fe_valor_ajuizado"],
    apply_s=df_dev["fe_valor_ajuizado"],
    lower_q=0.01,
    upper_q=0.99
)

df_holdout["fe_valor_ajuizado_winsor"], _, _ = winsorize_by_train_quantiles(
    train_s=df_dev["fe_valor_ajuizado"],
    apply_s=df_holdout["fe_valor_ajuizado"],
    lower_q=0.01,
    upper_q=0.99
)

df_dev["fe_valor_ajuizado_winsor_log1p"] = safe_log1p(df_dev["fe_valor_ajuizado_winsor"])
df_holdout["fe_valor_ajuizado_winsor_log1p"] = safe_log1p(df_holdout["fe_valor_ajuizado_winsor"])

winsor_report = pd.DataFrame([{
    "feature": "fe_valor_ajuizado",
    "lower_q": 0.01,
    "upper_q": 0.99,
    "lower_value_fit_dev": lower_winsor,
    "upper_value_fit_dev": upper_winsor,
}])

save_report(winsor_report, "06_winsor_limits_fit_on_dev.csv")
winsor_report

## 17. Consolidar lista de features early-stage v1

In [ ]:
numeric_features = [
    "fe_valor_ajuizado",
    "fe_valor_ajuizado_log1p",
    "fe_valor_ajuizado_winsor",
    "fe_valor_ajuizado_winsor_log1p",
    "fe_flag_valor_ate_10k",
    "fe_flag_valor_10k_50k",
    "fe_flag_valor_acima_50k",
    "fe_flag_valor_acima_100k",
] + temporal_features + quality_features + keyword_features

categorical_features = (
    ["fe_faixa_valor_ajuizado"] +
    categorical_norm_features +
    interaction_features
)

text_model_features = [
    "fe_texto_inicial_curto"
]

all_features_early_v1 = []

for col in numeric_features + categorical_features + text_model_features:
    if col in df_dev.columns and col not in all_features_early_v1:
        all_features_early_v1.append(col)

feature_dict = pd.DataFrame({
    "feature": all_features_early_v1,
    "tipo": [
        "numeric" if f in numeric_features else
        "categorical" if f in categorical_features else
        "text"
        for f in all_features_early_v1
    ],
    "dtype_dev": [str(df_dev[f].dtype) for f in all_features_early_v1],
    "missing_rate_dev": [df_dev[f].isna().mean() for f in all_features_early_v1],
    "nunique_dev": [df_dev[f].nunique(dropna=True) for f in all_features_early_v1],
})

save_report(feature_dict, "07_feature_dictionary_early_v1.csv")

print("Qtd features early_v1:", len(all_features_early_v1))
feature_dict

## 18. Validação final de leakage nas features

In [ ]:
feature_leakage_check = []

for feature in all_features_early_v1:
    leak = is_leakage_col(feature) or feature in blocked_early_stage_cols_found

    feature_leakage_check.append({
        "feature": feature,
        "possivel_leakage": leak
    })

feature_leakage_check = pd.DataFrame(feature_leakage_check)

save_report(feature_leakage_check, "08_feature_leakage_check_early_v1.csv")

feature_leakage_check["possivel_leakage"].value_counts()

Se aparecer `True`, investigue antes de prosseguir.

In [ ]:
feature_leakage_check[feature_leakage_check["possivel_leakage"] == True]

## 19. Salvar datasets da Etapa 1

In [ ]:
cols_id = existing_cols(df_dev, [
    "Numerodistribuicao",
    "Identificador",
    DATE_COL,
    TARGET_COL,
])

cols_to_save = cols_id + all_features_early_v1

# Remover duplicatas preservando ordem
cols_to_save = list(dict.fromkeys(cols_to_save))

df_dev_out = df_dev[cols_to_save].copy()
df_holdout_out = df_holdout[cols_to_save].copy()

dev_path = PROCESSED_DIR / "abt_early_stage_v1_dev.parquet"
holdout_path = PROCESSED_DIR / "abt_early_stage_v1_holdout.parquet"
full_path = PROCESSED_DIR / "abt_early_stage_v1_full_rowwise.parquet"

df_dev_out.to_parquet(dev_path, index=False)
df_holdout_out.to_parquet(holdout_path, index=False)

pd.concat([df_dev_out, df_holdout_out], axis=0).to_parquet(full_path, index=False)

print("Dev salvo em:", dev_path, df_dev_out.shape)
print("Holdout salvo em:", holdout_path, df_holdout_out.shape)
print("Full salvo em:", full_path)

## 20. Salvar lista de features

In [ ]:
feature_list_path = PROCESSED_DIR / "feature_list_early_stage_v1.txt"

with open(feature_list_path, "w", encoding="utf-8") as f:
    for feature in all_features_early_v1:
        f.write(feature + "\n")

print("Lista de features salva em:", feature_list_path)

## 21. Resumo final da Etapa 1

In [ ]:
summary_step_1 = pd.DataFrame([{
    "qtd_linhas_original_validada": len(df),
    "qtd_linhas_dev": len(df_dev_out),
    "qtd_linhas_holdout": len(df_holdout_out),
    "qtd_features_early_v1": len(all_features_early_v1),
    "target_rate_dev": df_dev_out[TARGET_COL].mean(),
    "target_rate_holdout": df_holdout_out[TARGET_COL].mean(),
    "split_date": split_date,
    "dev_path": str(dev_path),
    "holdout_path": str(holdout_path),
    "feature_list_path": str(feature_list_path),
}])

save_report(summary_step_1, "09_summary_step_1_prepare_early_stage_abt.csv")
summary_step_1

# O que verificar após rodar

Traga para a próxima iteração:

1. Shape inicial e shape após validações.
2. Distribuição do target.
3. Quantidade de colunas removidas por leakage.
4. Quantidade de colunas bloqueadas para early-stage.
5. Split report:
   - `split_date`
   - `qtd_dev`
   - `qtd_holdout`
   - `taxa_target_dev`
   - `taxa_target_holdout`
6. Quantidade final de features.
7. Se apareceu alguma feature com `possivel_leakage = True`.

Se esta etapa estiver correta, a próxima será:

## Etapa 2 — Features históricas sem leakage

Nela criaremos:

- taxa histórica suavizada por assunto/produto/ação;
- count histórico;
- is_rare histórico;
- valor mediano histórico por grupo;
- ratio valor atual / mediana histórica;
- transformers com `fit` apenas no dev/train e `transform` no holdout.